In [2]:
from elasticsearch import Elasticsearch

In [3]:
# 1. Kết nối Elasticsearch
es = Elasticsearch("http://localhost:9200", verify_certs=False)


In [40]:
index_name = "customers"

settings = {
    "settings": {
        "analysis": {
            "filter": {
                "edge_ngram_filter": {
                    "type": "edge_ngram",
                    "min_gram": 1,
                    "max_gram": 5
                }
            },
            "analyzer": {
                "custom_name_analyzer": {
                    "type": "custom",
                    "tokenizer": "standard",
                    "filter": ["lowercase", "edge_ngram_filter"]
                }
            }
        }
    },
    "mappings": {
        "properties": {
            "name": {
                "type": "text",
                "analyzer": "custom_name_analyzer",
                "search_analyzer": "standard"
            },
            "address": {"type": "text"},
            "age": {"type": "integer"}
        }
    }
}

from elasticsearch import Elasticsearch

es = Elasticsearch("http://localhost:9200", verify_certs=False)

# Xóa nếu đã tồn tại
if es.indices.exists(index=index_name):
    es.indices.delete(index=index_name)

# Tạo index mới
es.indices.create(index=index_name, body=settings)

ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'customers'})

In [60]:
docs = [
    {"name": "Nguyen Van Anh", "address": "Ha Noi", "age": 30},
    {"name": "Nguyen Van A", "address": "HCM", "age": 25},
    {"name": "Ng V A", "address": "Da Nang", "age": 40},
    {"name": "Tran Thi B", "address": "Hue", "age": 27},
    {"name": "Nguyen Van C", "address": "Hai Phong", "age": 22}
]

for doc in docs:
    es.index(index=index_name, document=doc)

In [61]:
tokens = es.indices.analyze(
    index=index_name,
    body={
        "analyzer": "custom_name_analyzer",
        "text": "Nguyen Van Anh"
    }
)

print("📌 Tokens:", [t["token"] for t in tokens["tokens"]])

📌 Tokens: ['n', 'ng', 'ngu', 'nguy', 'nguye', 'v', 'va', 'van', 'a', 'an', 'anh']


In [69]:
query_string = "Ng V A"

query_body = {
    "query": {
        "match": {
            "name": {
                "query": query_string,
                "fuzziness": "Auto",
                "operator": "and"
            }
        }
    }
}

res = es.search(index=index_name, body=query_body)
for hit in res["hits"]["hits"]:
    print(hit["_source"])

{'name': 'Nguyen Van Anh', 'address': 'Ha Noi', 'age': 30}
{'name': 'Nguyen Van A', 'address': 'HCM', 'age': 25}
{'name': 'Ng V A', 'address': 'Da Nang', 'age': 40}
{'name': 'Nguyen Van Anh', 'address': 'Ha Noi', 'age': 30}
{'name': 'Nguyen Van A', 'address': 'HCM', 'age': 25}
{'name': 'Ng V A', 'address': 'Da Nang', 'age': 40}


In [52]:
# 5. Tìm kiếm fuzzy (gần đúng tên)
query_string = "Nguyễn Văn A"

query_body = {
    "query": {
        "match": {
            "name": {
                "query": query_string,
                "fuzziness": "Auto",
                # "operator": "or"
            }
        }
    }
}

res = es.search(index=index_name, body=query_body)

# 6. Hiển thị kết quả
print(f"🔍 Kết quả fuzzy search cho: '{query_string}'")
for hit in res["hits"]["hits"]:
    print(hit["_source"])

🔍 Kết quả fuzzy search cho: 'Nguyễn Văn A'
{'name': 'Nguyen Van Anh', 'address': 'Ha Noi', 'age': 30}
{'name': 'Nguyen Van A', 'address': 'HCM', 'age': 25}
{'name': 'Ng V A', 'address': 'Da Nang', 'age': 40}


In [6]:
import pandas as pd
from rapidfuzz import process, fuzz

# Tạo dữ liệu mẫu
data = {
    'full_name': [
        'Nguyễn Minh Hiếu',
        'Nguyễn Mạnh Hùng',
        'Trần Văn An',
        'Minh Hiếu',
        'N M H',
        'Nguyễn Văn Hiếu',
        'Nguyễn Thị Hiền',
        'Ngô Minh Hải',
        'Nguyễn Văn Hòa',
        'Hiếu Nguyễn Minh'
    ]
}
df = pd.DataFrame(data)

# Hàm tìm tên tương tự chỉ dùng fuzzy
def search_similar_names_fuzzy_only(query, df, column='full_name', limit=5, threshold=70):
    # Fuzzy match toàn bộ dữ liệu
    results = process.extract(query, df[column], scorer=fuzz.ratio, limit=limit)

    # Lọc theo ngưỡng điểm tương đồng
    results = [(name, score) for name, score, _ in results if score >= threshold]

    # Trả kết quả có độ giống
    return df[df[column].isin([name for name, _ in results])].assign(similarity=[s for _, s in results])

# Test
query_name = "Nguyễn Minh Hiếu"
result_df = search_similar_names_fuzzy_only(query_name, df)

# Hiển thị kết quả
print(f"Top tên giống với '{query_name}':")
print(result_df)

Top tên giống với 'Nguyễn Minh Hiếu':
          full_name  similarity
0  Nguyễn Minh Hiếu  100.000000
1  Nguyễn Mạnh Hùng   83.870968
3         Minh Hiếu   75.000000
5   Nguyễn Văn Hiếu   72.000000
7      Ngô Minh Hải   71.428571


In [4]:
from sentence_transformers import SentenceTransformer, util
import pandas as pd
import numpy as np
import torch


# 1. Khởi tạo model
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

# 2. Dữ liệu tên mẫu
names = [
    'Nguyễn Minh Hiếu',
    'Nguyễn Mạnh Hùng',
    'Trần Văn An',
    'Minh Hiếu',
    'N M H',
    'Nguyễn Văn Hiếu',
    'Nguyễn Thị Hiền',
    'Ngô Minh Hải',
    'Nguyễn Văn Hòa',
    'Hiếu Nguyễn Minh'
]

df = pd.DataFrame({'full_name': names})

# 3. Encode tất cả tên thành embedding
print("🔍 Encoding all names...")
embeddings = model.encode(df['full_name'].tolist(), convert_to_tensor=True)

# 4. Hàm tìm tên giống nhất với truy vấn
def search_similar_names(query, top_k=5, threshold=0.7):
    query_emb = model.encode(query, convert_to_tensor=True)
    cosine_scores = util.cos_sim(query_emb, embeddings)[0]

    # Lấy top-k theo điểm cosine
    top_results = torch.topk(cosine_scores, k=top_k)

    results = []
    for score, idx in zip(top_results.values, top_results.indices):
        score_val = score.item()
        idx_val = idx.item()  # ✅ chuyển về int
        if score_val >= threshold:
            results.append({
                "name": df.iloc[idx_val]['full_name'],
                "score": round(score_val, 4)
            })
    return results

# 5. Test truy vấn
query_name = "Nguyễn Minh Hiếu"
results = search_similar_names(query_name, top_k=5, threshold=0.5)

# 6. In kết quả
print(f"\n🔎 Top tên giống với '{query_name}':")
for res in results:
    print(f"{res['name']:25s}  →  similarity: {res['score']}")

🔍 Encoding all names...

🔎 Top tên giống với 'Nguyễn Minh Hiếu':
Nguyễn Minh Hiếu           →  similarity: 1.0
Hiếu Nguyễn Minh           →  similarity: 0.9844
Minh Hiếu                  →  similarity: 0.9667
Nguyễn Mạnh Hùng           →  similarity: 0.9139
Nguyễn Thị Hiền            →  similarity: 0.9028
